In [2]:
import pandas as pd
import numpy as np
import json
import seaborn as sns
import matplotlib.pyplot as plt

# Warning off
pd.options.mode.chained_assignment = None

import warnings
warnings.filterwarnings("ignore")

In [3]:
df = pd.read_csv('/home/sunaybhat/results_PureDefense/From_Scratch/Narcissus/Results.csv')

# Last 10 rows of df
df_filt = df.iloc[-9:,:]

print(f'Nat Acc Avg: {df_filt["End Acc"].mean():.2%} | Poison Acc Avg: {df_filt["P1 Acc"].mean():.2%} | Max Poison Acc: {df_filt["P1 Acc"].max():.2%}')


Nat Acc Avg: 75.41% | Poison Acc Avg: 2.56% | Max Poison Acc: 5.30%


## Testing HLB

In [16]:
df = pd.read_csv('/home/sunaybhat/results_PureDefense/Clean/Results.csv')
df['hlb_type'] = df['Args'].apply(lambda x: json.loads(x)['hlb_type'] if 'hlb_type' in x else np.nan)
df['Model'] = df.apply(lambda x: x['Model'] + ' ' + x['hlb_type'] if x['Model'] == 'HLB' else x['Model'], axis=1)

results = []
for i in df['Model'].unique():
    df_filt = df[(df['Model'] == i)]
    nat_acc_avg = df_filt.iloc[-8:,:]["End Acc"].mean()
    train_time_avg = df_filt.iloc[-8:,:]["Train Time"].mean()
    results.append({'Model': i, 'Nat Acc (%))': nat_acc_avg*100, 'Train Time (s))': train_time_avg})

results_df = pd.DataFrame(results)
print(results_df.to_latex(index=False, float_format="%.2f"))

\begin{tabular}{lrr}
\toprule
Model & Nat Acc (%)) & Train Time (s)) \\
\midrule
HLB small & 92.52 & 168.44 \\
HLB medium & 93.25 & 237.73 \\
HLB large & 94.96 & 864.48 \\
ResNet18_HLB & 93.86 & 615.37 \\
ResNet18 & 94.90 & 4879.38 \\
\bottomrule
\end{tabular}



In [24]:
df = pd.read_csv('/home/sunaybhat/results_PureDefense/From_Scratch/Narcissus/Results.csv')
df['hlb_type'] = df['Args'].apply(lambda x: json.loads(x)['hlb_type'] if 'hlb_type' in x else np.nan)
df['Model'] = df.apply(lambda x: x['Model'] + ' ' + x['hlb_type'] if x['Model'] == 'HLB' else x['Model'], axis=1)
df['Eps'] = df['Args'].apply(lambda x: json.loads(x)['noise_eps_narcissus'] if 'noise_eps_narcissus' in x else np.nan)

results = {}
for j in df['Eps'].unique():
    results[j] = {}
    for i in df['Model'].unique():
        df_filt = df[(df['Model'] == i) & (df['Eps'] == j)]
        nat_acc_avg = df_filt["End Acc"].mean()
        nat_acc_std = df_filt["End Acc"].std()
        poison_acc_avg = df_filt["P1 Acc"].mean()
        poison_acc_std = df_filt["P1 Acc"].std()
        results[j][i] = {
            'Nat Acc Avg': f"{nat_acc_avg*100:.2f} ± {nat_acc_std*100:.2f}",
            'Poison Acc Avg': f"{poison_acc_avg*100:.2f} ± {poison_acc_std*100:.2f}",
            'Max Poison Acc': df_filt["P1 Acc"].max()*100
        }
results_df = pd.DataFrame(results).T.stack().apply(pd.Series)
print(results_df.to_latex(float_format="%.2f"))

\begin{tabular}{llllr}
\toprule
 &  & Nat Acc Avg & Poison Acc Avg & Max Poison Acc \\
\midrule
\multirow[t]{5}{*}{8} & HLB small & 92.57 ± 0.19 & 12.10 ± 19.43 & 62.12 \\
 & HLB medium & 93.45 ± 0.14 & 15.53 ± 21.41 & 69.87 \\
 & HLB large & 95.02 ± 0.11 & 31.89 ± 33.91 & 80.75 \\
 & ResNet18_HLB & 93.55 ± 0.12 & 28.94 ± 31.26 & 84.36 \\
 & ResNet18 & 94.92 ± 0.11 & 39.66 ± 31.28 & 85.67 \\
\cline{1-5}
\multirow[t]{5}{*}{16} & HLB small & 92.47 ± 0.10 & 44.23 ± 19.28 & 74.44 \\
 & HLB medium & 93.48 ± 0.21 & 47.57 ± 19.38 & 80.43 \\
 & HLB large & 94.96 ± 0.13 & 71.92 ± 14.22 & 93.73 \\
 & ResNet18_HLB & 93.62 ± 0.10 & 64.37 ± 17.54 & 85.10 \\
 & ResNet18 & 94.90 ± 0.13 & 78.55 ± 10.35 & 92.72 \\
\cline{1-5}
\bottomrule
\end{tabular}



In [20]:
results_df

Nat Acc Avg Poison Acc Avg  Max Poison Acc
8  HLB small     0.93 ± 0.00    0.12 ± 0.19        0.621215
   HLB medium    0.93 ± 0.00    0.16 ± 0.21        0.698702
   HLB large     0.95 ± 0.00    0.32 ± 0.34        0.807548
   ResNet18_HLB  0.94 ± 0.00    0.29 ± 0.31        0.843640
   ResNet18      0.95 ± 0.00    0.40 ± 0.31        0.856712
16 HLB small     0.92 ± 0.00    0.44 ± 0.19        0.744388
   HLB medium    0.93 ± 0.00    0.48 ± 0.19        0.804291
   HLB large     0.95 ± 0.00    0.72 ± 0.14        0.937280
   ResNet18_HLB  0.94 ± 0.00    0.64 ± 0.18        0.851012
   ResNet18      0.95 ± 0.00    0.79 ± 0.10        0.927245

In [6]:
df = pd.read_csv('/home/sunaybhat/results_PureDefense/From_Scratch/Narcissus/Results.csv')
df['hlb_type'] = df['Args'].apply(lambda x: json.loads(x)['hlb_type'] if 'hlb_type' in x else np.nan)
df['Model'] = df.apply(lambda x: x['Model'] + ' ' + x['hlb_type'] if x['Model'] == 'HLB' else x['Model'], axis=1)
df['Eps'] = df['Args'].apply(lambda x: json.loads(x)['noise_eps_narcissus'] if 'noise_eps_narcissus' in x else np.nan)

for j in df['Eps'].unique():
    print(f'\nNarciussus Poison Eps: {j}\n____________________')
    for i in df['Model'].unique():
        df_filt = df[(df['Model'] == i) & a(df['Eps'] == j)]
        print(f'Model: {i} | Nat Acc Avg: {df_filt["End Acc"].mean():.2%} | Poison Acc Avg: {df_filt["P1 Acc"].mean():.2%} | Max Poison Acc: {df_filt["P1 Acc"].max():.2%}')



Narciussus Poison Eps: 8
____________________
Model: HLB small | Nat Acc Avg: 92.57% | Poison Acc Avg: 12.10% | Max Poison Acc: 62.12%
Model: HLB medium | Nat Acc Avg: 93.45% | Poison Acc Avg: 15.53% | Max Poison Acc: 69.87%
Model: HLB large | Nat Acc Avg: 95.02% | Poison Acc Avg: 31.89% | Max Poison Acc: 80.75%
Model: ResNet18_HLB | Nat Acc Avg: 93.55% | Poison Acc Avg: 28.94% | Max Poison Acc: 84.36%
Model: ResNet18 | Nat Acc Avg: 94.92% | Poison Acc Avg: 39.66% | Max Poison Acc: 85.67%

Narciussus Poison Eps: 16
____________________
Model: HLB small | Nat Acc Avg: 92.47% | Poison Acc Avg: 44.23% | Max Poison Acc: 74.44%
Model: HLB medium | Nat Acc Avg: 93.48% | Poison Acc Avg: 47.57% | Max Poison Acc: 80.43%
Model: HLB large | Nat Acc Avg: 94.96% | Poison Acc Avg: 71.92% | Max Poison Acc: 93.73%
Model: ResNet18_HLB | Nat Acc Avg: 93.62% | Poison Acc Avg: 64.37% | Max Poison Acc: 85.10%
Model: ResNet18 | Nat Acc Avg: 94.90% | Poison Acc Avg: 78.55% | Max Poison Acc: 92.72%


### Hyperparameter viewing

In [16]:
import pandas as pd
import numpy as np
import json
import re

In [24]:
# Load the CSV file
df = pd.read_csv('/home/alexbranch/results_PureDefense/From_Scratch/Narcissus/Results.csv')

# Preprocess and extract additional information
df['hlb_type'] = df['Args'].apply(lambda x: json.loads(x)['hlb_type'] if 'hlb_type' in x else np.nan)
df['Model'] = df.apply(lambda x: x['Model'] + ' ' + x['hlb_type'] if x['Model'] == 'HLB' else x['Model'], axis=1)
df['Eps'] = df['Args'].apply(lambda x: json.loads(x)['noise_eps_narcissus'] if 'noise_eps_narcissus' in x else np.nan)

# Debugging extraction function: Let's print a sample data key to understand its format
print("Sample 'data key':", df['Data Key'].iloc[0])

def extract_steps(s):
    ebm_steps_match = re.search(r'\{(\d+)\}Steps', s)
    diffusion_steps_matches = re.findall(r'\{(\d+)\}Steps', s)
    ebm_steps = int(ebm_steps_match.group(1)) if ebm_steps_match else np.nan
    # Assuming the second match is for diffusion steps; adjusting index if necessary.
    diffusion_steps = int(diffusion_steps_matches[1]) if len(diffusion_steps_matches) > 1 else np.nan
    return ebm_steps, diffusion_steps

# Apply the function and add the new columns to the DataFrame
df['EBM_Steps'], df['Diffusion_Steps'] = zip(*df['Data Key'].apply(extract_steps))

# Let's print out the unique values of EBM and Diffusion steps to ensure they're extracted correctly.
print("Unique EBM Steps:", df['EBM_Steps'].unique())
print("Unique Diffusion Steps:", df['Diffusion_Steps'].unique())

KeyError: 'data key'

In [21]:

# Analysis
results = []

for eps in df['Eps'].unique():
    for model in df['Model'].unique():
        for ebm_steps in df['EBM_Steps'].unique():
            for diffusion_steps in df['Diffusion_Steps'].unique():
                df_filt = df[(df['Model'] == model) & (df['Eps'] == eps) & (df['EBM_Steps'] == ebm_steps) & (df['Diffusion_Steps'] == diffusion_steps)]
                if not df_filt.empty:
                    results.append({
                        'Eps': eps,
                        'Model': model,
                        'EBM_Steps': ebm_steps,
                        'Diffusion_Steps': diffusion_steps,
                        'Nat Acc Avg': df_filt["End Acc"].mean(),
                        'Poison Acc Avg': df_filt["P1 Acc"].mean(),
                        'Max Poison Acc': df_filt["P1 Acc"].max(),
                    })

# Convert results to DataFrame for better visualization
results_df = pd.DataFrame(results)
print(results)

[]


In [22]:
print(results_df)


Empty DataFrame
Columns: []
Index: []


## Old Results

In [3]:
def process_filtered_df(results_df,df_def,defense,narcissus,header,header_idx):
    if narcissus:
        avg_poison_success = df_def['P1 Acc'].mean() * 100
        std_poison_success = df_def['P1 Acc'].std() * 100
        avg_nat_acc = df_def['End Acc'].mean()
        std_nat_acc = df_def['End Acc'].std()
        max_poison_success = df_def['P1 Acc'].max() * 100


        if header is None:
            results_df.loc[defense, 'Avg Poison Success'] = f'{avg_poison_success:.2f}\u00B1{std_poison_success:.1f}'
            results_df.loc[defense,'Avg Nat Acc'] = f'{avg_nat_acc:.2f}\u00B1{std_nat_acc:.2f}'
            results_df.loc[defense,  'Max Poison Success'] = f'{max_poison_success:.2f}'

        else:
            results_df.loc[defense, (header_idx, 'Avg Poison Success')] = f'{avg_poison_success:.2%}\u00B1{std_poison_success:.1f}'
            results_df.loc[defense, (header_idx, 'Avg Nat Acc')] = f'{avg_nat_acc:.2f}\u00B1{std_nat_acc:.1f}'
            results_df.loc[defense, (header_idx, 'Max Poison Success')] = f'{max_poison_success:.2f}'

    else:

        success_rate = (df_def["Success"] == True).sum() / len(df_def)
        avg_nat_acc = df_def["End Acc"].mean()
        std_nat_acc = df_def["End Acc"].std()

        if header is None:
            results_df.loc[defense, 'Poison Success'] = f'{success_rate:.2%}'
            results_df.loc[defense, 'Natural Accuracy'] = f'{avg_nat_acc:.2f}%\u00B1{std_nat_acc:.1f}'

        else:
            results_df.loc[defense, (header_idx, 'Poison Success')] = f'{success_rate:.2%}'
            results_df.loc[defense, (header_idx, 'Natural Accuracy')] = f'{avg_nat_acc:.2f}%\u00B1{std_nat_acc:.1f}'

def export_results(df, narcissus = False, header=None, filters = {}):

    # Iterate through the filters
    for key, value in filters.items():
        df = df[df[key] == value]

    # Initialize an empty DataFrame for results
    if narcissus:
        if header is None:
            columns = ['Avg Poison Success', 'Avg Nat Acc','Max Poison Success']
        else:
            columns = pd.MultiIndex.from_product([np.sort(df[header].unique()), ['Avg Poison Success', 'Avg Nat Acc','Max Poison Success']],
                                     names=[header, 'Metric'])

    else:
        if header is None:
            columns = ['Poison Success', 'Natural Accuracy']
        else:
            columns = pd.MultiIndex.from_product([df[header].unique(), ['Poison Success', 'Natural Accuracy']],
                                                names=[header, 'Metric'])

    # Define the defenses
    defenses = ['None', 'EBM','Diff','EBM_Diff']                                         
    
    # Populate the DataFrame
    results_df = pd.DataFrame(columns=columns)

    if header is None: header_idxs = [0]
    else: header_idxs = df[header].unique()

    for header_idx in header_idxs:

        for defense in defenses:
            if header is None:
                if defense == 'Diff':
                    for steps in np.sort(df['Steps'].unique()):
                        df_def = df[(df['Defense'] == defense) & (df['Steps'] == steps)]
                        process_filtered_df(results_df,df_def,defense+f'-{steps}',narcissus,header,header_idx)
                        print(f'Added {defense}-{steps} to results_df with num rows {len(df_def)}')
                elif defense == 'EBM_Diff':
                    for steps in np.sort(df['Steps'].unique()):
                        df_def = df[(df['Defense'] == defense) & (df['Steps'] == steps)]
                        process_filtered_df(results_df,df_def,defense+f'-{steps}',narcissus,header,header_idx)
                        print(f'Added {defense}-{steps} to results_df with num rows {len(df_def)}')
                else:
                    df_def = df[(df['Defense'] == defense)]
                    process_filtered_df(results_df,df_def,defense,narcissus,header,header_idx)
                    print(f'Added {defense} to results_df with num rows {len(df_def)}')

            else:
                df_def = df[(df['Defense'] == 'None') & (df[header] == header_idx)]
                process_filtered_df(results_df,df_def,defense,narcissus,header,header_idx)

    return results_df

In [4]:
df = pd.read_csv('/home/sunaybhat/results_EBM_Defense/From_Scratch/Narcissus/Results.csv')

df['Defense'] = df['Defense'].fillna('None')
df['Steps'] = df['subset_size'] = df['Args'].apply(lambda x: json.loads(x)['diff_steps'] if 'diff_steps' in x else np.nan)
df['Steps'] = df['Steps'].fillna(125)

df_results = export_results(df,narcissus=True)



Added None to results_df with num rows 10
Added EBM to results_df with num rows 10
Added Diff-10.0 to results_df with num rows 10
Added Diff-25.0 to results_df with num rows 10
Added Diff-50.0 to results_df with num rows 10
Added Diff-100.0 to results_df with num rows 10
Added Diff-125.0 to results_df with num rows 10
Added Diff-150.0 to results_df with num rows 10
Added Diff-175.0 to results_df with num rows 10
Added Diff-200.0 to results_df with num rows 10
Added Diff-250.0 to results_df with num rows 10
Added Diff-500.0 to results_df with num rows 10
Added EBM_Diff-10.0 to results_df with num rows 10
Added EBM_Diff-25.0 to results_df with num rows 10
Added EBM_Diff-50.0 to results_df with num rows 10
Added EBM_Diff-100.0 to results_df with num rows 10
Added EBM_Diff-125.0 to results_df with num rows 10
Added EBM_Diff-150.0 to results_df with num rows 10
Added EBM_Diff-175.0 to results_df with num rows 10
Added EBM_Diff-200.0 to results_df with num rows 10
Added EBM_Diff-250.0 to res

,Avg Poison Success,Avg Nat Acc,Max Poison Success
None,43.95±33.6,94.89±0.23,93.59
EBM,1.34±0.6,92.73±0.22,2.29
Diff-10.0,42.63±47.5,94.42±0.13,99.76
Diff-25.0,42.08±48.6,94.04±0.16,99.99
Diff-50.0,22.66±39.8,93.67±0.13,99.86
Diff-100.0,2.36±2.4,93.64±0.13,8.92
Diff-125.0,2.00±2.0,93.50±0.16,7.49
Diff-150.0,1.96±1.9,93.36±0.23,7.16
Diff-175.0,1.72±1.4,93.24±0.26,5.42
Diff-200.0,1.40±0.9,93.19±0.29,3.69


In [9]:
df['Logs'].apply(lambda x: json.loads(x)['train_loss'])[0]


[1.9179745479617887,
 1.4543709767139172,
 1.2407012272368916,
 1.0339778805022959,
 0.8892892341479621,
 0.7750748780834705,
 0.6866544254905428,
 0.627800113481024,
 0.5800987233591202,
 0.5474209582714169,
 0.5197739152194899,
 0.5031422742492403,
 0.4824650273146227,
 0.4665756484736567,
 0.44813149755873033,
 0.4442681756318378,
 0.43076425786975703,
 0.4244910193526227,
 0.41725032142056223,
 0.40715148069364643,
 0.4011640531937485,
 0.39429829000969374,
 0.38584966835615886,
 0.3810331524942842,
 0.3743264714394079,
 0.37221626152315407,
 0.36462125003032975,
 0.3605127510664713,
 0.3563685262066019,
 0.35731240348590304,
 0.34913173569437794,
 0.3489763981012432,
 0.34767301510209625,
 0.34432335826746946,
 0.34254109905199015,
 0.33746281491063745,
 0.3349005826522627,
 0.33470999283711317,
 0.32959385349622466,
 0.3272460234896911,
 0.3294612658603112,
 0.32370188302548647,
 0.32619049074247364,
 0.3233782731930313,
 0.3207666279790956,
 0.319612363415301,
 0.315635062373050